# Nematode classifiers

Train AlexNet, VGG16, and ResNet-50 on dataset_inema/train. Validate only on dataset_inema/val.

In [8]:
%pip install -q timm pillow pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from pathlib import Path

ROOT = Path('/home/boibhai/GitHub/nematode-detection')
TRAIN_DIR = ROOT / 'dataset_inema' / 'train'
VAL_DIR = ROOT / 'dataset_inema' / 'val'
PRETRAINED_DIR = ROOT / 'pretrained_models'
TRAINED_DIR = ROOT / 'trained_models'
PRETRAINED_DIR.mkdir(exist_ok=True)
TRAINED_DIR.mkdir(exist_ok=True)

# torchvision and timm store downloaded pretrained weights here, not in a temporary kernel cache.
os.environ['TORCH_HOME'] = str(PRETRAINED_DIR)
os.environ['HF_HOME'] = str(PRETRAINED_DIR / 'huggingface')

import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = 224
BATCH_SIZE = 2       # VGG16 fits safely on this 3.68 GB GPU at this batch size.
EPOCHS = 10
LR = 1e-4
print('Device:', DEVICE)

Device: cuda


In [10]:
# Create five rotated copies of every original training image.
# Existing *_rot_ copies are ignored, so rerunning this cell does not make duplicates.
random.seed(42)
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
originals = [p for p in TRAIN_DIR.rglob('*') if p.suffix.lower() in extensions and '_rot_' not in p.stem]

for source in originals:
    with Image.open(source) as image:
        image = ImageOps.exif_transpose(image).convert('RGB')
        for copy_number in range(1, 2):
            target = source.with_name(f'{source.stem}_rot_{copy_number}.jpg')
            if not target.exists():
                angle = random.uniform(-180, 180)
                image.rotate(angle, resample=Image.Resampling.BICUBIC).save(target, quality=95)

print(f'Original images: {len(originals)}')
print(f'Training images after rotation: {sum(1 for p in TRAIN_DIR.rglob("*") if p.suffix.lower() in extensions)}')
print(f'Validation images unchanged: {sum(1 for p in VAL_DIR.rglob("*") if p.suffix.lower() in extensions)}')

Original images: 2217
Training images after rotation: 4434
Validation images unchanged: 552


In [11]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((.485, .456, .406), (.229, .224, .225)),
])

train_set = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_set = datasets.ImageFolder(VAL_DIR, transform=transform)
assert train_set.classes == val_set.classes, 'Train and validation class folders must match.'
classes = train_set.classes

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_test_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(len(classes), 'classes |', len(train_set), 'train images |', len(val_set), 'validation images')

19 classes | 4434 train images | 552 validation images


In [12]:
# Run once. Downloads all three original ImageNet weights into pretrained_models when it is empty.
if not any(PRETRAINED_DIR.iterdir()):
    models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    timm.create_model('vgg16.tv_in1k', pretrained=True)
    timm.create_model('resnet50', pretrained=True)
    print('Downloaded pretrained models to:', PRETRAINED_DIR)
else:
    print('Using pretrained models already in:', PRETRAINED_DIR)

Using pretrained models already in: /home/boibhai/GitHub/nematode-detection/pretrained_models


In [13]:
def clear_gpu():
    # Releases models left in notebook variables between separate training runs.
    for value in list(globals().values()):
        if isinstance(value, nn.Module):
            value.to('cpu')
    gc.collect()
    torch.cuda.empty_cache()

def make_alexnet():
    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(classes))
    return model

def make_vgg16():
    return timm.create_model('vgg16.tv_in1k', pretrained=True, num_classes=len(classes))

def make_resnet50():
    return timm.create_model('resnet50', pretrained=True, num_classes=len(classes))

def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for images, labels in loader:
            predictions = model(images.to(DEVICE)).argmax(1).cpu()
            correct += (predictions == labels).sum().item()
            total += len(labels)
    return correct / total

def train(model, name):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    best_accuracy = 0
    best_weights = None

    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = loss_fn(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        validation_accuracy = accuracy(model, val_loader)
        print(f'{name} epoch {epoch + 1}/{EPOCHS}: validation accuracy = {validation_accuracy:.3f}')
        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_weights)
    return model, best_accuracy

def save_model(model, name):
    path = TRAINED_DIR / f'{name}_nematode.pth'
    torch.save({'classes': classes, 'state_dict': model.state_dict()}, path)
    print('Saved:', path)

In [14]:
# AlexNet: run this cell independently.
clear_gpu()
alexnet, alexnet_val = train(make_alexnet(), 'AlexNet')
alexnet_train = accuracy(alexnet, train_test_loader)
save_model(alexnet, 'alexnet')
print(f'AlexNet train={alexnet_train:.3f}, validation={alexnet_val:.3f}')
alexnet.to('cpu')
del alexnet
torch.cuda.empty_cache()

AlexNet epoch 1/10: validation accuracy = 0.433
AlexNet epoch 2/10: validation accuracy = 0.426
AlexNet epoch 3/10: validation accuracy = 0.458
AlexNet epoch 4/10: validation accuracy = 0.531
AlexNet epoch 5/10: validation accuracy = 0.422
AlexNet epoch 6/10: validation accuracy = 0.489
AlexNet epoch 7/10: validation accuracy = 0.525
AlexNet epoch 8/10: validation accuracy = 0.507
AlexNet epoch 9/10: validation accuracy = 0.493
AlexNet epoch 10/10: validation accuracy = 0.533
Saved: /home/boibhai/GitHub/nematode-detection/trained_models/alexnet_nematode.pth
AlexNet train=0.964, validation=0.533


In [15]:
# VGG16: run this cell independently.
clear_gpu()
vgg16, vgg16_val = train(make_vgg16(), 'VGG16')
vgg16_train = accuracy(vgg16, train_test_loader)
save_model(vgg16, 'vgg16')
print(f'VGG16 train={vgg16_train:.3f}, validation={vgg16_val:.3f}')
vgg16.to('cpu')
del vgg16
torch.cuda.empty_cache()

RuntimeError: Unknown model (vgg16.tv_in1k)

In [ ]:
# ResNet-50: run this cell independently.
clear_gpu()
resnet50, resnet50_val = train(make_resnet50(), 'ResNet-50')
resnet50_train = accuracy(resnet50, train_test_loader)
save_model(resnet50, 'resnet50')
print(f'ResNet-50 train={resnet50_train:.3f}, validation={resnet50_val:.3f}')
resnet50.to('cpu')
del resnet50
torch.cuda.empty_cache()

In [ ]:
comparison = pd.DataFrame([
    ['AlexNet', alexnet_train, alexnet_val],
    ['VGG16', vgg16_train, vgg16_val],
    ['ResNet-50', resnet50_train, resnet50_val],
], columns=['model', 'train_accuracy', 'validation_accuracy'])
comparison.to_csv(TRAINED_DIR / 'comparison.csv', index=False)
comparison.sort_values('validation_accuracy', ascending=False)

## Explainability: Grad-CAM

Grad-CAM highlights the image regions that raised a model's predicted class score. Use it to confirm predictions are based on nematode morphology rather than labels, borders, or other artifacts.

In [ ]:
# Set this to the model you want to inspect after it has been trained.
MODEL_FOR_EXPLANATION = 'resnet50'
model_builders = {'alexnet': make_alexnet, 'vgg16': make_vgg16, 'resnet50': make_resnet50}

explain_model = model_builders[MODEL_FOR_EXPLANATION]().to(DEVICE)
checkpoint = torch.load(TRAINED_DIR / f'{MODEL_FOR_EXPLANATION}_nematode.pth', map_location=DEVICE, weights_only=False)
explain_model.load_state_dict(checkpoint['state_dict'])
target_layer = [layer for layer in explain_model.modules() if isinstance(layer, nn.Conv2d)][-1]
saved = {}
def save_features(module, inputs, output):
    saved['activations'] = output
    output.register_hook(lambda gradient: saved.update({'gradients': gradient}))
target_layer.register_forward_hook(save_features)

image, label = val_set[0]
scores = explain_model(image.unsqueeze(0).to(DEVICE))
prediction = scores.argmax(1).item()
explain_model.zero_grad()
scores[0, prediction].backward()

weights = saved['gradients'].mean(dim=(2, 3), keepdim=True)
cam = torch.relu((weights * saved['activations']).sum(1, keepdim=True))
cam = torch.nn.functional.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)[0, 0]
cam = cam.detach().cpu().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

image_np = image.permute(1, 2, 0).numpy() * np.array((.229, .224, .225)) + np.array((.485, .456, .406))
image_np = np.clip(image_np, 0, 1)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(image_np); plt.title(f'True: {classes[label]}'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(image_np); plt.imshow(cam, cmap='jet', alpha=.45); plt.title(f'Predicted: {classes[prediction]}'); plt.axis('off')
plt.tight_layout()